# Adjusted Transparent Lifecycle Order with Severity

This notebook builds a severity distribution table for the adjusted transparent lifecycle-order patterns.

Inputs:
- `rq1_transparent_lifecycle_order_adjusted.ipynb` for `transparent_lifecycle_order_adjusted_df` and `selected_adjusted_lifecycle_order_df`.
- `repo_with_severity.csv` for CVE severity.

No CSV is saved by default.

In [8]:
import contextlib
import io
from pathlib import Path

import nbformat
import pandas as pd

In [9]:
#

In [10]:
ADJUSTED_NOTEBOOK_PATH = Path('../../data/rq1/transparent_data/rq1_transparent_lifecycle_order_adjusted.ipynb')
SEVERITY_PATH = Path('../../data/rq1/repo_with_severity.csv')

severity = pd.read_csv(SEVERITY_PATH)
print('severity:', severity.shape, 'unique CVEs:', severity['CVE_ID'].nunique())
severity['Severity'].value_counts(dropna=False)

severity: (832, 10) unique CVEs: 832


Severity
Medium      608
High        140
Low          47
Critical     36
Unknown       1
Name: count, dtype: int64

## Load Adjusted Transparent Lifecycle Data

This executes the adjusted lifecycle notebook in memory so this notebook uses the latest manual corrections and exclusions.

In [11]:
adjusted_nb = nbformat.read(ADJUSTED_NOTEBOOK_PATH, as_version=4)
adjusted_ns = {}

# Execute quietly to avoid repeating all intermediate outputs from the adjusted-order notebook.
captured_output = io.StringIO()
with contextlib.redirect_stdout(captured_output):
    for i, cell in enumerate(adjusted_nb.cells):
        if cell.cell_type == 'code':
            exec(compile(cell.source, f'adjusted_notebook_cell_{i}', 'exec'), adjusted_ns)

transparent_lifecycle_order_adjusted_df = adjusted_ns['transparent_lifecycle_order_adjusted_df'].copy()
transparent_lifecycle_summary_adjusted = adjusted_ns['transparent_lifecycle_summary_adjusted'].copy()
manual_excluded_adjusted_df = adjusted_ns.get('manual_excluded_adjusted_df', pd.DataFrame()).copy()

print('adjusted lifecycle rows:', len(transparent_lifecycle_order_adjusted_df))
print('manual excluded rows:', len(manual_excluded_adjusted_df))
transparent_lifecycle_summary_adjusted

adjusted lifecycle rows: 278
manual excluded rows: 1


,First,Second,Third,Fourth,Count,%
0,Report,Fix,Release,Disclosure,216,77.70
1,Report,Fix,Disclosure,Release,26,9.35
2,Disclosure,Report,Fix,Release,13,4.68
3,Fix,Release,Disclosure,GAD,13,4.68
4,Report,Disclosure,Fix,Release,6,2.16
5,Disclosure,Fix,Release,GAD,2,0.72
6,Fix,Disclosure,Release,GAD,2,0.72


## Add Severity to Adjusted Lifecycle Data

In [12]:
severity_lookup = severity[['CVE_ID', 'Severity']].drop_duplicates(subset=['CVE_ID']).copy()

transparent_lifecycle_order_adjusted_with_severity_df = transparent_lifecycle_order_adjusted_df.merge(
    severity_lookup,
    on='CVE_ID',
    how='left',
)

print('Missing Severity in adjusted lifecycle data:', transparent_lifecycle_order_adjusted_with_severity_df['Severity'].isna().sum())
print('Severity counts in adjusted lifecycle data:')
print(transparent_lifecycle_order_adjusted_with_severity_df['Severity'].value_counts(dropna=False))

transparent_lifecycle_order_adjusted_with_severity_df.head()

Missing Severity in adjusted lifecycle data: 0
Severity counts in adjusted lifecycle data:
Severity
Medium      187
High         57
Low          20
Critical     14
Name: count, dtype: int64


,CVE_ID,repository,PATCH,PATCH_lifecycle,type_of_reporting,reporting_link,created_date_kind,Report Date,Fix Date,Release Date,...,Original Third,Original Fourth,First,Second,Third,Fourth,Original pattern,Adjusted pattern,Pattern changed,Severity
0,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,https://github.com/alkacon/opencms-core/commit...,github issue,https://github.com/alkacon/opencms-core/issues...,created_at,2013-06-13 21:26:01,2013-06-20 15:32:02,2013-07-09 11:57:20,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
1,CVE-2018-3831,elastic/elasticsearch,https://github.com/elastic/elasticsearch/commi...,https://github.com/elastic/elasticsearch/commi...,github pull request,https://github.com/elastic/elasticsearch/pull/...,created_at,2018-08-29 16:33:41,2018-08-29 16:31:56,2018-08-29 16:31:56,...,Report,Disclosure,Report,Fix,Release,Disclosure,"(Fix, Release, Report, Disclosure)","(Report, Fix, Release, Disclosure)",True,Medium
2,CVE-2019-16943,FasterXML/jackson-databind,https://github.com/FasterXML/jackson-databind/...,https://github.com/FasterXML/jackson-databind/...,github issue,https://github.com/FasterXML/jackson-databind/...,created_at,2019-09-27 15:44:21,2019-09-29 12:12:38,2019-11-09 15:11:27,...,Disclosure,Release,Report,Fix,Disclosure,Release,"(Report, Fix, Disclosure, Release)","(Report, Fix, Disclosure, Release)",False,Medium
3,CVE-2021-37714,jhy/jsoup,https://github.com/jhy/jsoup/commit/d2c455c94a...,https://github.com/jhy/jsoup/commit/d2c455c94a...,github issue,https://github.com/jhy/jsoup/issues/1613,created_at,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
4,CVE-2020-36282,rabbitmq/rabbitmq-jms-client,https://github.com/rabbitmq/rabbitmq-jms-clien...,https://github.com/rabbitmq/rabbitmq-jms-clien...,github issue,https://github.com/rabbitmq/rabbitmq-jms-clien...,created_at,2020-11-02 10:38:12,2020-11-02 11:39:24,2020-11-03 09:27:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High


In [18]:
transparent_lifecycle_order_adjusted_with_severity_df

,CVE_ID,repository,PATCH,PATCH_lifecycle,type_of_reporting,reporting_link,created_date_kind,Report Date,Fix Date,Release Date,...,Original Third,Original Fourth,First,Second,Third,Fourth,Original pattern,Adjusted pattern,Pattern changed,Severity
0,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,https://github.com/alkacon/opencms-core/commit...,github issue,https://github.com/alkacon/opencms-core/issues...,created_at,2013-06-13 21:26:01,2013-06-20 15:32:02,2013-07-09 11:57:20,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
1,CVE-2018-3831,elastic/elasticsearch,https://github.com/elastic/elasticsearch/commi...,https://github.com/elastic/elasticsearch/commi...,github pull request,https://github.com/elastic/elasticsearch/pull/...,created_at,2018-08-29 16:33:41,2018-08-29 16:31:56,2018-08-29 16:31:56,...,Report,Disclosure,Report,Fix,Release,Disclosure,"(Fix, Release, Report, Disclosure)","(Report, Fix, Release, Disclosure)",True,Medium
2,CVE-2019-16943,FasterXML/jackson-databind,https://github.com/FasterXML/jackson-databind/...,https://github.com/FasterXML/jackson-databind/...,github issue,https://github.com/FasterXML/jackson-databind/...,created_at,2019-09-27 15:44:21,2019-09-29 12:12:38,2019-11-09 15:11:27,...,Disclosure,Release,Report,Fix,Disclosure,Release,"(Report, Fix, Disclosure, Release)","(Report, Fix, Disclosure, Release)",False,Medium
3,CVE-2021-37714,jhy/jsoup,https://github.com/jhy/jsoup/commit/d2c455c94a...,https://github.com/jhy/jsoup/commit/d2c455c94a...,github issue,https://github.com/jhy/jsoup/issues/1613,created_at,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
4,CVE-2020-36282,rabbitmq/rabbitmq-jms-client,https://github.com/rabbitmq/rabbitmq-jms-clien...,https://github.com/rabbitmq/rabbitmq-jms-clien...,github issue,https://github.com/rabbitmq/rabbitmq-jms-clien...,created_at,2020-11-02 10:38:12,2020-11-02 11:39:24,2020-11-03 09:27:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
273,CVE-2018-1258,spring-projects/spring-security,https://github.com/spring-projects/spring-secu...,https://github.com/spring-projects/spring-secu...,github issue,https://github.com/spring-projects/spring-secu...,created_at,2018-04-24 09:39:25,2018-05-07 16:23:32,2018-05-08 09:22:58,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
274,CVE-2020-13946,apache/cassandra,https://github.com/apache/cassandra/commit/63f...,https://github.com/apache/cassandra/commit/63f...,apache issue tracker,https://issues.apache.org/jira/browse/CASSANDR...,created_at,2020-08-27 16:20:00,2020-08-28 09:32:40,2020-08-28 13:16:01,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
275,CVE-2020-11989,apache/shiro,https://github.com/apache/shiro/commit/b90f918...,https://github.com/apache/shiro/commit/b90f918...,apache issue tracker,https://issues.apache.org/jira/browse/SHIRO-753,created_at,2020-04-07 09:41:00,2020-04-07 15:56:56,2020-04-26 20:42:18,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
276,CVE-2016-1181,kawasima/struts1-forever,https://github.com/kawasima/struts1-forever/co...,https://github.com/kawasima/struts1-forever/co...,github advisory,https://github.com/advisories/GHSA-7jw3-5q4w-89qg,published_at,2022-05-13 01:25:20,2016-06-08 11:38:30,2016-06-08 11:38:30,...,Disclosure,Report,Fix,Release,Disclosure,GAD,"(Fix, Release, Disclosure,

## Transparent Lifecycle Order Severity Table

In [13]:
severity_order = ['Critical', 'High', 'Medium', 'Low', 'Unknown']


def format_severity_distribution(values):
    counts = values.value_counts(dropna=False).to_dict()
    parts = []

    for severity_label in severity_order:
        count = int(counts.get(severity_label, 0))
        if severity_label == 'Unknown':
            if count > 0:
                parts.append(f'{severity_label} ({count})')
        else:
            parts.append(f'{severity_label} ({count})')

    for severity_label, count in sorted(counts.items(), key=lambda item: str(item[0])):
        if severity_label not in severity_order:
            parts.append(f'{severity_label} ({int(count)})')

    return ', '.join(parts)


transparent_lifecycle_order_severity_table = (
    transparent_lifecycle_order_adjusted_with_severity_df
    .groupby(['First', 'Second', 'Third', 'Fourth'])
    .agg(
        Count=('CVE_ID', 'count'),
        **{'Severity distribution': ('Severity', format_severity_distribution)}
    )
    .reset_index()
)

transparent_lifecycle_order_severity_table['%'] = (
    transparent_lifecycle_order_severity_table['Count']
    / len(transparent_lifecycle_order_adjusted_with_severity_df)
    * 100
).round(2)

transparent_lifecycle_order_severity_table = transparent_lifecycle_order_severity_table[
    ['First', 'Second', 'Third', 'Fourth', 'Count', '%', 'Severity distribution']
].sort_values('Count', ascending=False).reset_index(drop=True)

transparent_lifecycle_order_severity_table

,First,Second,Third,Fourth,Count,%,Severity distribution
0,Report,Fix,Release,Disclosure,216,77.70,"Critical (10), High (42), Medium (147), Low (17)"
1,Report,Fix,Disclosure,Release,26,9.35,"Critical (1), High (7), Medium (16), Low (2)"
2,Disclosure,Report,Fix,Release,13,4.68,"Critical (2), High (5), Medium (6), Low (0)"
3,Fix,Release,Disclosure,GAD,13,4.68,"Critical (1), High (2), Medium (9), Low (1)"
4,Report,Disclosure,Fix,Release,6,2.16,"Critical (0), High (1), Medium (5), Low (0)"
5,Disclosure,Fix,Release,GAD,2,0.72,"Critical (0), High (0), Medium (2), Low (0)"
6,Fix,Disclosure,Release,GAD,2,0.72,"Critical (0), High (0), Medium (2), Low (0)"


## Severity Counts as Columns

In [14]:
transparent_lifecycle_order_severity_wide = (
    transparent_lifecycle_order_adjusted_with_severity_df
    .pivot_table(
        index=['First', 'Second', 'Third', 'Fourth'],
        columns='Severity',
        values='CVE_ID',
        aggfunc='count',
        fill_value=0,
    )
    .reset_index()
)

for severity_label in severity_order:
    if severity_label not in transparent_lifecycle_order_severity_wide.columns:
        transparent_lifecycle_order_severity_wide[severity_label] = 0

transparent_lifecycle_order_severity_wide['Count'] = transparent_lifecycle_order_severity_wide[severity_order].sum(axis=1)
transparent_lifecycle_order_severity_wide['%'] = (
    transparent_lifecycle_order_severity_wide['Count']
    / len(transparent_lifecycle_order_adjusted_with_severity_df)
    * 100
).round(2)

transparent_lifecycle_order_severity_wide = transparent_lifecycle_order_severity_wide[
    ['First', 'Second', 'Third', 'Fourth', 'Count', '%'] + severity_order
].sort_values('Count', ascending=False).reset_index(drop=True)

transparent_lifecycle_order_severity_wide

Severity,First,Second,Third,Fourth,Count,%,Critical,High,Medium,Low,Unknown
0,Report,Fix,Release,Disclosure,216,77.70,10,42,147,17,0
1,Report,Fix,Disclosure,Release,26,9.35,1,7,16,2,0
2,Disclosure,Report,Fix,Release,13,4.68,2,5,6,0,0
3,Fix,Release,Disclosure,GAD,13,4.68,1,2,9,1,0
4,Report,Disclosure,Fix,Release,6,2.16,0,1,5,0,0
5,Disclosure,Fix,Release,GAD,2,0.72,0,0,2,0,0
6,Fix,Disclosure,Release,GAD,2,0.72,0,0,2,0,0


## Inspect Selected Adjusted Lifecycle Order with Severity

Change `ORDER_TO_INSPECT` to any adjusted lifecycle-order tuple shown in `transparent_lifecycle_order_severity_table`.

In [15]:
ORDER_TO_INSPECT = ('Report', 'Fix', 'Release', 'Disclosure')

order_keys = transparent_lifecycle_order_adjusted_with_severity_df[
    ['First', 'Second', 'Third', 'Fourth']
].apply(lambda row: tuple(row), axis=1)

selected_adjusted_lifecycle_order_with_severity_df = transparent_lifecycle_order_adjusted_with_severity_df[
    order_keys.map(lambda key: key == ORDER_TO_INSPECT)
].copy()

print('Selected adjusted lifecycle order:', ORDER_TO_INSPECT)
print('Rows:', len(selected_adjusted_lifecycle_order_with_severity_df))
print('Severity distribution:')
print(selected_adjusted_lifecycle_order_with_severity_df['Severity'].value_counts(dropna=False))

selected_adjusted_lifecycle_order_with_severity_df

Selected adjusted lifecycle order: ('Report', 'Fix', 'Release', 'Disclosure')
Rows: 216
Severity distribution:
Severity
Medium      147
High         42
Low          17
Critical     10
Name: count, dtype: int64


,CVE_ID,repository,PATCH,PATCH_lifecycle,type_of_reporting,reporting_link,created_date_kind,Report Date,Fix Date,Release Date,...,Original Third,Original Fourth,First,Second,Third,Fourth,Original pattern,Adjusted pattern,Pattern changed,Severity
0,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,https://github.com/alkacon/opencms-core/commit...,github issue,https://github.com/alkacon/opencms-core/issues...,created_at,2013-06-13 21:26:01,2013-06-20 15:32:02,2013-07-09 11:57:20,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
1,CVE-2018-3831,elastic/elasticsearch,https://github.com/elastic/elasticsearch/commi...,https://github.com/elastic/elasticsearch/commi...,github pull request,https://github.com/elastic/elasticsearch/pull/...,created_at,2018-08-29 16:33:41,2018-08-29 16:31:56,2018-08-29 16:31:56,...,Report,Disclosure,Report,Fix,Release,Disclosure,"(Fix, Release, Report, Disclosure)","(Report, Fix, Release, Disclosure)",True,Medium
3,CVE-2021-37714,jhy/jsoup,https://github.com/jhy/jsoup/commit/d2c455c94a...,https://github.com/jhy/jsoup/commit/d2c455c94a...,github issue,https://github.com/jhy/jsoup/issues/1613,created_at,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
4,CVE-2020-36282,rabbitmq/rabbitmq-jms-client,https://github.com/rabbitmq/rabbitmq-jms-clien...,https://github.com/rabbitmq/rabbitmq-jms-clien...,github issue,https://github.com/rabbitmq/rabbitmq-jms-clien...,created_at,2020-11-02 10:38:12,2020-11-02 11:39:24,2020-11-03 09:27:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
5,CVE-2017-15696,apache/geode,https://github.com/apache/geode/commit/37a8970...,https://github.com/apache/geode/commit/37a8970...,github pull request,https://github.com/apache/geode/pull/1059,created_at,2017-11-14 21:40:20,2017-11-21 07:59:38,2018-01-26 08:34:31,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,CVE-2020-11998,apache/activemq,https://github.com/apache/activemq/commit/0d6e...,https://github.com/apache/activemq/commit/0d6e...,apache issue tracker,https://issues.apache.org/jira/browse/AMQ-7490,created_at,2020-05-22 08:56:00,2020-05-23 06:38:58,2020-05-25 07:58:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
273,CVE-2018-1258,spring-projects/spring-security,https://github.com/spring-projects/spring-secu...,https://github.com/spring-projects/spring-secu...,github issue,https://github.com/spring-projects/spring-secu...,created_at,2018-04-24 09:39:25,2018-05-07 16:23:32,2018-05-08 09:22:58,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
274,CVE-2020-13946,apache/cassandra,https://github.com/apache/cassandra/commit/63f...,https://github.com/apache/cassandra/commit/63f...,apache issue tracker,https://issues.apache.org/jira/browse/CASSANDR...,created_at,2020-08-27 16:20:00,2020-08-28 09:32:40,2020-08-28 13:16:01,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
275,CVE-2020-11989,apache/shiro,https://github.com/apache/shiro/commit/b90f918...,https://github.com/apache/shiro/commit/b90f918...,apache issue tracker,https://issues.apache.org/jira/browse/SHIRO-753,created_at,2020-04-07 09:41:00,2020-04-07 15:56:56,2020-04-26 20:42:18,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report,

In [16]:
selected_adjusted_lifecycle_order_with_severity_df

,CVE_ID,repository,PATCH,PATCH_lifecycle,type_of_reporting,reporting_link,created_date_kind,Report Date,Fix Date,Release Date,...,Original Third,Original Fourth,First,Second,Third,Fourth,Original pattern,Adjusted pattern,Pattern changed,Severity
0,CVE-2013-4600,alkacon/opencms-core,https://github.com/alkacon/opencms-core/commit...,https://github.com/alkacon/opencms-core/commit...,github issue,https://github.com/alkacon/opencms-core/issues...,created_at,2013-06-13 21:26:01,2013-06-20 15:32:02,2013-07-09 11:57:20,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
1,CVE-2018-3831,elastic/elasticsearch,https://github.com/elastic/elasticsearch/commi...,https://github.com/elastic/elasticsearch/commi...,github pull request,https://github.com/elastic/elasticsearch/pull/...,created_at,2018-08-29 16:33:41,2018-08-29 16:31:56,2018-08-29 16:31:56,...,Report,Disclosure,Report,Fix,Release,Disclosure,"(Fix, Release, Report, Disclosure)","(Report, Fix, Release, Disclosure)",True,Medium
3,CVE-2021-37714,jhy/jsoup,https://github.com/jhy/jsoup/commit/d2c455c94a...,https://github.com/jhy/jsoup/commit/d2c455c94a...,github issue,https://github.com/jhy/jsoup/issues/1613,created_at,2021-08-15 02:25:46,2021-08-15 12:31:12,2021-08-15 15:39:15,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
4,CVE-2020-36282,rabbitmq/rabbitmq-jms-client,https://github.com/rabbitmq/rabbitmq-jms-clien...,https://github.com/rabbitmq/rabbitmq-jms-clien...,github issue,https://github.com/rabbitmq/rabbitmq-jms-clien...,created_at,2020-11-02 10:38:12,2020-11-02 11:39:24,2020-11-03 09:27:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
5,CVE-2017-15696,apache/geode,https://github.com/apache/geode/commit/37a8970...,https://github.com/apache/geode/commit/37a8970...,github pull request,https://github.com/apache/geode/pull/1059,created_at,2017-11-14 21:40:20,2017-11-21 07:59:38,2018-01-26 08:34:31,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
272,CVE-2020-11998,apache/activemq,https://github.com/apache/activemq/commit/0d6e...,https://github.com/apache/activemq/commit/0d6e...,apache issue tracker,https://issues.apache.org/jira/browse/AMQ-7490,created_at,2020-05-22 08:56:00,2020-05-23 06:38:58,2020-05-25 07:58:56,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,High
273,CVE-2018-1258,spring-projects/spring-security,https://github.com/spring-projects/spring-secu...,https://github.com/spring-projects/spring-secu...,github issue,https://github.com/spring-projects/spring-secu...,created_at,2018-04-24 09:39:25,2018-05-07 16:23:32,2018-05-08 09:22:58,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
274,CVE-2020-13946,apache/cassandra,https://github.com/apache/cassandra/commit/63f...,https://github.com/apache/cassandra/commit/63f...,apache issue tracker,https://issues.apache.org/jira/browse/CASSANDR...,created_at,2020-08-27 16:20:00,2020-08-28 09:32:40,2020-08-28 13:16:01,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report, Fix, Release, Disclosure)",False,Medium
275,CVE-2020-11989,apache/shiro,https://github.com/apache/shiro/commit/b90f918...,https://github.com/apache/shiro/commit/b90f918...,apache issue tracker,https://issues.apache.org/jira/browse/SHIRO-753,created_at,2020-04-07 09:41:00,2020-04-07 15:56:56,2020-04-26 20:42:18,...,Release,Disclosure,Report,Fix,Release,Disclosure,"(Report, Fix, Release, Disclosure)","(Report,

In [ ]:
# No CSV is saved by default. Uncomment only if you need exports later.
# transparent_lifecycle_order_adjusted_with_severity_df.to_csv(DATA_DIR / 'transparent_lifecycle_order_adjusted_with_severity.csv', index=False)
# transparent_lifecycle_order_severity_table.to_csv(DATA_DIR / 'transparent_lifecycle_order_adjusted_severity_table.csv', index=False)
# transparent_lifecycle_order_severity_wide.to_csv(DATA_DIR / 'transparent_lifecycle_order_adjusted_severity_wide.csv', index=False)